In [ ]:
%pip install datasets

In [ ]:
from datasets import load_dataset
wiki = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)
wiki_sample = wiki.shuffle(seed=42).take(200_000)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

In [ ]:
with open("wiki_500mb.txt", "w", encoding="utf8") as f:
    for row in wiki_sample:
        f.write(row["text"] + "\n")


In [ ]:
import re
from tqdm import tqdm

def is_bad_line(line):
    """Check if line should be skipped."""
    line_lower = line.lower()

    # Section headers
    if re.match(r'^=+.*=+$', line):
        return True

    # Wiki markup
    if line.startswith('{{') or line.startswith('}}'):
        return True
    if line.startswith('[[Category:') or line.startswith('[[File:'):
        return True

    # List items
    if line.startswith('*') or line.startswith('#'):
        return True

    # Boilerplate sections
    skip_words = ['references', 'see also', 'external links',
                  'further reading', 'bibliography', 'notes']
    if line_lower in skip_words:
        return True

    return False

def clean_line(line):
    """Clean inline wiki markup from a line."""
    # [[link|text]] -> text
    line = re.sub(r'\[\[([^|\]]+\|)?([^\]]+)\]\]', r'\2', line)
    # Remove templates {{...}}
    line = re.sub(r'\{\{[^}]+\}\}', '', line)
    # Remove references
    line = re.sub(r'<ref[^>]*>.*?</ref>', '', line, flags=re.DOTALL)
    line = re.sub(r'<ref[^>]*/>', '', line)
    # Remove HTML tags
    line = re.sub(r'<[^>]+>', '', line)
    # Remove URLs
    line = re.sub(r'https?://\S+', '', line)
    # Remove coordinates
    line = re.sub(r'\d+°\d+[′\']?\d*[″\"]?[NSEW]', '', line)
    # Normalize whitespace
    line = re.sub(r'\s+', ' ', line).strip()

    return line

def clean_article(text):
    """Clean a single article."""
    lines = text.split("\n")
    cleaned = []

    for line in lines:
        line = line.strip()
        if not line:
            continue
        if is_bad_line(line):
            continue
        if 'may refer to' in line.lower():
            return None  # Skip disambiguation pages

        line = clean_line(line)
        if len(line) > 20:
            cleaned.append(line)

    result = ' '.join(cleaned)
    result = re.sub(r'\s+', ' ', result).strip()

    if len(result) < 200:
        return None

    return result

# Process the file
print("Loading file...")
with open('wiki_500mb.txt', 'r', encoding='utf8') as f:
    content = f.read()

# Split into articles (each article starts on a new line after empty lines)
print("Splitting into articles...")
articles = re.split(r'\n{2,}', content)  # Split on 2+ newlines

print(f"Found {len(articles)} raw articles")

# Clean articles
print("Cleaning articles...")
cleaned = []
for article in tqdm(articles):
    result = clean_article(article)
    if result:
        cleaned.append(result)

print(f"Kept {len(cleaned)} cleaned articles")

# Save
with open('wiki_cleaned.txt', 'w', encoding='utf8') as f:
    for article in cleaned:
        f.write(article + '\n\n')

print("Done! Saved to wiki_cleaned.txt")

Loading file...
Splitting into articles...
Found 1765883 raw articles
Cleaning articles...


100%|██████████| 1765883/1765883 [01:52<00:00, 15728.04it/s]


Kept 749975 cleaned articles
Done! Saved to wiki_cleaned.txt


In [ ]:
import torch
from torch import nn
from torch.nn import Sequential
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2TokenizerFast
from tqdm import tqdm
import torch.optim as optim
device = 'cuda' if torch.cuda.is_available() else 'cpu'


In [ ]:
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
all_tokens=[]
print("Tokenizing...")
with open("wiki_cleaned.txt", "r", encoding='utf8') as f:
	for line in tqdm(f):
		line = line.strip()
		if line:
			tokens = tokenizer.encode(line, add_special_tokens=False)
			all_tokens.extend(tokens)
print(f"Total tokens: {len(all_tokens):,}")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Tokenizing...


2143it [00:00, 5095.98it/s]Token indices sequence length is longer than the specified maximum sequence length for this model (2943 > 1024). Running this sequence through the model will result in indexing errors
1499950it [05:39, 4423.31it/s]

Total tokens: 88,723,485


In [ ]:
chunk_size = 512
print("Creating chunks...")
chunks = []

for i in range(0, len(all_tokens) - chunk_size, chunk_size):
	chunks.append(all_tokens[i : i+chunk_size])

print(f"Total chunks: {len(chunks):,}")

Creating chunks...
Total chunks: 173,288


In [ ]:
def casual_mask(dim, device):
	return torch.triu(torch.ones(dim, dim, device=device), diagonal=1).bool()

class Block(nn.Module):
	def __init__(self):
		super().__init__()
		"""Attention layer 256-> embedding/model dimensions, 8-> Attention heads,
		batch_first=True-> format will be batch, seq, feature"""
		self.attention = nn.MultiheadAttention(256, 8, batch_first=True)

		"""Feed forward Layer have 256 dim input and hidden layer with 1024 Nodes
		-> GeLU() applied -> Output layer with 256 Nodes
		"""
		self.ff_network = nn.Sequential(
			nn.Linear(256, 1024),
			nn.GELU(),
			nn.Linear(1024, 256)
		)

		"""Layer Normalization layers have 256 dimensions to be normalized of input
		(output of attention layer and Feed Forward layer)
		"""
		self.layer_norm1 = nn.LayerNorm(256)
		self.layer_norm2 = nn.LayerNorm(256)
	# Defining forward pass
	def forward(self, x):
		# getting shape of input embeddings(_ is embedding dimensions)
		batch_size, sequence_length, _ = x.shape

		# getting mask for masked attention
		mask = casual_mask(sequence_length, x.device)

		# Attention output
		attn_out, _ = self.attention(x, x, x, attn_mask=mask)
		# Layer Norm 1
		x = self.layer_norm1(x + attn_out)
		# Feed Forward Layer output
		ff_out = self.ff_network(x)
		# Layer Norm2
		x = self.layer_norm2(x + ff_out)
		# Retrun the output
		return x



In [ ]:
class MiniGPT(nn.Module):
	def __init__(self, vocab_size, device=device):
		super().__init__()
		# Token embeddings, input is 50157(vocabulary size) and output is 256 (embedding dimensions)
		self.token_embeddings = nn.Embedding(vocab_size, 256, device=device)
		# Postional embeddings have sequence length as input and output is 256 (pos_embedding dimensions)
		self.pos_embeddings = nn.Embedding(512, 256, device=device)
		# Creating 6 Blocks of decoder
		self.blocks = nn.ModuleList([Block() for _ in range(6)])
		# Layer normalization layer
		self.layer_norm = nn.LayerNorm(256)
		# This Feed Forward head is for final prediction input will be 256 dim and output will be 50157(score for all tokens in vocabulary)
		self.head = nn.Linear(256, vocab_size)

	def forward(self, x):
		batch_size, sequence_length = x.shape

		pos = torch.arange(sequence_length).unsqueeze(0).to(x.device)
		x = self.token_embeddings(x) + self.pos_embeddings(pos)
		for block in self.blocks:
			x = block(x)

		x = self.layer_norm(x)
		return self.head(x)

In [ ]:
class MiniGPTDataset(Dataset):
	def __init__(self, chunks):
		self.data = chunks

	def __len__(self):
		return len(self.data)

	def __getitem__(self, idx):
		x = torch.tensor(self.data[idx][:-1])
		y = torch.tensor(self.data[idx][1:])
		return x, y

In [ ]:
dataset = MiniGPTDataset(chunks=chunks)

loader = DataLoader(dataset, batch_size=8, pin_memory=True, shuffle=True)

In [ ]:
# Initializing model
gpt_model = MiniGPT(tokenizer.vocab_size)
gpt_model = gpt_model.to(device)

# Loss Function
loss_fn = nn.CrossEntropyLoss()

# Using Adam Optimizer
optimizer = optim.Adam(gpt_model.parameters(), lr=0.01)

In [ ]:
import time
def generate_text(model, tokenizer, prompt, max_length=50, device='cuda'):
    """Generate text from the model given a prompt"""
    model.eval()
    tokens = tokenizer.encode(prompt)
    input_ids = torch.tensor([tokens]).to(device)

    with torch.no_grad():
        for _ in range(max_length):
            # Get predictions
            logits = model(input_ids)
            # Get the last token's logits
            next_token_logits = logits[0, -1, :]
            # Sample from the distribution (you can use argmax for greedy)
            next_token = torch.multinomial(torch.softmax(next_token_logits, dim=-1), num_samples=1)
            # Append to sequence
            input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)
            # Truncate if exceeds max seq length
            if input_ids.shape[1] > 511:
                input_ids = input_ids[:, -511:]

    model.train()
    return tokenizer.decode(input_ids[0].tolist())

In [ ]:
gpt_model.train()
epochs = 3
log_interval = 500  # Har 500 steps par loss print
sample_prompt = "The history of"
print(f"Training on device: {device}")
print(f"Total batches per epoch: {len(loader)}")
print("-" * 50)
for epoch in range(epochs):
    total_loss = 0
    epoch_start = time.time()

    for step, (x, y) in enumerate(tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}")):
        # Moving data to device
        x, y = x.to(device), y.to(device)

        # Forward Pass
        logits = gpt_model(x)

        # Reshape for CrossEntropyLoss
        batch_size, seq_len, vocab_size = logits.shape
        logits = logits.view(batch_size * seq_len, vocab_size)
        y = y.view(batch_size * seq_len)

        # Calculate loss
        loss = loss_fn(logits, y)

        # Zero grads, backward, update
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Log loss every log_interval steps
        if (step + 1) % log_interval == 0:
            avg_loss = total_loss / (step + 1)
            print(f"\n[Step {step+1}/{len(loader)}] Loss: {loss.item():.4f} | Avg Loss: {avg_loss:.4f}")

    # Epoch summary
    epoch_time = time.time() - epoch_start
    avg_epoch_loss = total_loss / len(loader)
    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/{epochs} Complete!")
    print(f"Average Loss: {avg_epoch_loss:.4f}")
    print(f"Time: {epoch_time/60:.2f} minutes")
    print(f"{'='*50}")

    # sample text after each epoch
    print(f"\n📝 Sample Generation (prompt: '{sample_prompt}'):")
    generated = generate_text(gpt_model, tokenizer, sample_prompt, max_length=50, device=device)
    print(f"Generated: {generated}")
    print("-" * 50)
print("\n🎉 Training Complete!")

Training on device: cuda
Total batches per epoch: 21661
--------------------------------------------------


Epoch 1/3:   2%|▏         | 501/21661 [01:40<1:10:39,  4.99it/s]


[Step 500/21661] Loss: 8.1559 | Avg Loss: 8.0080


Epoch 1/3:   5%|▍         | 1001/21661 [03:20<1:08:57,  4.99it/s]


[Step 1000/21661] Loss: 7.9129 | Avg Loss: 7.9534


...

Epoch 3/3:  67%|██████▋   | 14501/21661 [48:04<23:40,  5.04it/s]


[Step 14500/21661] Loss: 7.9002 | Avg Loss: 7.8748


Epoch 3/3:  69%|██████▉   | 15001/21661 [49:43<22:03,  5.03it/s]


[Step 15000/21661] Loss: 7.8777 | Avg Loss: 7.8749


Epoch 3/3:  70%|███████   | 15211/21661 [50:25<21:23,  5.03it/s]